In [2]:
# Install on Colab (if needed)
# !pip install cudf-cu12 cuml-cu12 seaborn -q

In [6]:
import numpy as np
from numba import cuda
import math

print("CUDA Available:", cuda.is_available())

@cuda.jit
def square_star_kernel(pattern_matrix):
    # Get the 2D position (thread index) for this specific GPU thread
    row, col = cuda.grid(2)

    n = pattern_matrix.shape[0]

    # Ensure threads don't execute outside the bounds of our matrix
    if row < n and col < n:
        # ASCII 42 = '*', ASCII 32 = ' '

        # Rule 1: Draw the square border
        if row == 0 or row == n - 1 or col == 0 or col == n - 1:
            pattern_matrix[row, col] = 42

        # Rule 2: Draw the inner star/cross (diagonals)
        elif row == col or row == n - 1 - col:
            pattern_matrix[row, col] = 42

        # Rule 3: Fill the rest with blank spaces
        else:
            pattern_matrix[row, col] = 32

# 1. Define the size of the square (N x N)
N = 15

# 2. Allocate an empty NumPy array of 8-bit integers (for ASCII)
# Initialize with spaces (32)
host_matrix = np.full((N, N), 32, dtype=np.uint8)

# 3. Transfer the array to the GPU (Device Memory)
device_matrix = cuda.to_device(host_matrix)

# 4. Configure Thread Blocks and Grids
# A standard 2D block size is 16x16 (256 threads per block)
threads_per_block = (16, 16)

# Calculate grid dimensions to ensure we cover the N x N matrix
blocks_per_grid_x = math.ceil(N / threads_per_block[0])
blocks_per_grid_y = math.ceil(N / threads_per_block[1])
blocks_per_grid = (blocks_per_grid_x, blocks_per_grid_y)

# 5. Launch the CUDA Kernel
square_star_kernel[blocks_per_grid, threads_per_block](device_matrix)

# 6. Wait for GPU to finish and copy the result back to the CPU
device_matrix.copy_to_host(host_matrix)

# 7. Decode the ASCII integers back to characters and print
print(f"--- GPU Generated Square Star Pattern ({N}x{N}) ---\n")
for row in host_matrix:
    # Convert ASCII codes back to characters and join them with spaces for visibility
    row_chars = [chr(code) for code in row]
    print(" ".join(row_chars))

CUDA Available: True


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


--- GPU Generated Square Star Pattern (15x15) ---

* * * * * * * * * * * * * * *
* *                       * *
*   *                   *   *
*     *               *     *
*       *           *       *
*         *       *         *
*           *   *           *
*             *             *
*           *   *           *
*         *       *         *
*       *           *       *
*     *               *     *
*   *                   *   *
* *                       * *
* * * * * * * * * * * * * * *
